In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('sample_data.csv')

In [3]:
df.head(5)

,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0


In [4]:
df.shape

(999, 17)

In [6]:
df.describe()

,amount (INR),fraud_flag,hour_of_day,is_weekend
count,999.000000,999.000000,999.000000,999.000000
mean,1294.501502,0.001001,15.083083,0.293293
std,1854.587274,0.031639,5.103940,0.455500
min,16.000000,0.000000,0.000000,0.000000
25%,282.500000,0.000000,11.000000,0.000000
50%,607.000000,0.000000,16.000000,0.000000
75%,1571.500000,0.000000,19.000000,1.000000
max,20396.000000,1.000000,23.000000,1.000000


In [7]:
df.columns

Index(['transaction id', 'timestamp', 'transaction type', 'merchant_category',
       'amount (INR)', 'transaction_status', 'sender_age_group',
       'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank',
       'device_type', 'network_type', 'fraud_flag', 'hour_of_day',
       'day_of_week', 'is_weekend'],
      dtype='str')

In [14]:
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

In [15]:
df.dtypes

transaction_id          str
timestamp               str
transaction_type        str
merchant_category       str
amount_(inr)          int64
transaction_status      str
sender_age_group        str
receiver_age_group      str
sender_state            str
sender_bank             str
receiver_bank           str
device_type             str
network_type            str
fraud_flag            int64
hour_of_day           int64
day_of_week             str
is_weekend            int64
dtype: object

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   transaction_id      999 non-null    str  
 1   timestamp           999 non-null    str  
 2   transaction_type    999 non-null    str  
 3   merchant_category   999 non-null    str  
 4   amount_(inr)        999 non-null    int64
 5   transaction_status  999 non-null    str  
 6   sender_age_group    999 non-null    str  
 7   receiver_age_group  999 non-null    str  
 8   sender_state        999 non-null    str  
 9   sender_bank         999 non-null    str  
 10  receiver_bank       999 non-null    str  
 11  device_type         999 non-null    str  
 12  network_type        999 non-null    str  
 13  fraud_flag          999 non-null    int64
 14  hour_of_day         999 non-null    int64
 15  day_of_week         999 non-null    str  
 16  is_weekend          999 non-null    int64
dtypes: int64

In [17]:
df.isnull().sum()

transaction_id        0
timestamp             0
transaction_type      0
merchant_category     0
amount_(inr)          0
transaction_status    0
sender_age_group      0
receiver_age_group    0
sender_state          0
sender_bank           0
receiver_bank         0
device_type           0
network_type          0
fraud_flag            0
hour_of_day           0
day_of_week           0
is_weekend            0
dtype: int64

In [18]:
#1. Check transaction_id uniqueness.
df['transaction_id'] = df['transaction_id'].nunique()

In [20]:
#2. Convert timestamp to datetime and create date/month/hour.
df['date'] = pd.to_datetime(df['timestamp']).dt.date
df['month'] = pd.to_datetime(df['timestamp']).dt.month
df['hour'] = pd.to_datetime(df['timestamp']).dt.hour

In [21]:
df.columns

Index(['transaction_id', 'timestamp', 'transaction_type', 'merchant_category',
       'amount_(inr)', 'transaction_status', 'sender_age_group',
       'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank',
       'device_type', 'network_type', 'fraud_flag', 'hour_of_day',
       'day_of_week', 'is_weekend', 'date', 'month', 'hour'],
      dtype='str')

In [22]:
#3. Check missing status, bank, device and amount fields.
df[['transaction_status', 'sender_bank', 'receiver_bank',
   'device_type', 'amount_(inr)']].isnull().sum().sum()

np.int64(0)

In [24]:
#4. Standardize status and payment-mode labels.
df['transaction_status'] = (
    df['transaction_status']
    .str.strip()
    .str.lower()
)

df['transaction_type'] = (
    df['transaction_type']
    .str.strip()
    .str.lower()
)
df[['transaction_status','transaction_type']].describe()

,transaction_status,transaction_type
count,999,999
unique,2,4
top,success,p2p
freq,953,446


In [29]:
# Check zero amounts
zero_amounts = (df['amount_(inr)'] == 0).sum()

# Check negative amounts
negative_amounts = (df['amount_(inr)'] < 0).sum()

# Check extreme amounts using IQR
Q1 = df['amount_(inr)'].quantile(0.25)
Q3 = df['amount_(inr)'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

extreme_amounts = (
    (df['amount_(inr)'] < lower_bound) |
    (df['amount_(inr)'] > upper_bound)
).sum()

print("Zero amounts:", zero_amounts)
print("Negative amounts:", negative_amounts)
print("Extreme amounts:", extreme_amounts)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Zero amounts: 0
Negative amounts: 0
Extreme amounts: 84
Lower bound: -1651.0
Upper bound: 3505.0


In [30]:
extreme_transactions = df[df['amount_(inr)'] > upper_bound]

print(extreme_transactions[['transaction_id', 'amount_(inr)']])

     transaction_id  amount_(inr)
36              999          7987
37              999          5154
41              999          4309
54              999          4987
79              999          8335
..              ...           ...
941             999          4844
947             999         11038
966             999          5736
976             999          7121
981             999          4804

[84 rows x 2 columns]


In [32]:
#6. Create failure_flag.
df['failure_flag'] = (
    df['transaction_status'].str.strip().str.lower() == 'failed'
).astype(int)

In [33]:
#7: Create amount buckets.
df['amount_bucket'] = pd.cut(
    df['amount_(inr)'],
    bins=[0, 500, 1500, 3505, float('inf')],
    labels=['Low', 'Medium', 'High', 'Extreme'],
    include_lowest=True
)

In [35]:
#8: Minimum transaction volume required for failure-rate comparisons
MIN_VOLUME = 30

print(f"Minimum volume threshold: {MIN_VOLUME} transactions")
print("Failure rates will only be compared for groups with at least "
      f"{MIN_VOLUME} transactions.")

Minimum volume threshold: 30 transactions
Failure rates will only be compared for groups with at least 30 transactions.


In [39]:
df['transaction_id'] = df['transaction_id'].astype(str).str.strip()

In [40]:
print(df['transaction_id'].head())
print(df['transaction_id'].dtype)

0    999
1    999
2    999
3    999
4    999
Name: transaction_id, dtype: str
str


In [46]:
df = pd.read_csv(
    'sample_data.csv',
    dtype={'transaction id': 'string'}
)

In [47]:
print(df['transaction id'].head())
print(df['transaction id'].dtype)

0    TXN0000000001
1    TXN0000000002
2    TXN0000000003
3    TXN0000000004
4    TXN0000000005
Name: transaction id, dtype: string
string


In [52]:
df["payment_method"] = df["transaction type"].str.lower().str.replace(" ", "_", regex=False)

In [55]:
df.columns = df.columns.str.replace(' ', '_', regex=False)

In [56]:
df.columns

Index(['transaction_id', 'timestamp', 'transaction_type', 'merchant_category',
       'amount_(INR)', 'transaction_status', 'sender_age_group',
       'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank',
       'device_type', 'network_type', 'fraud_flag', 'hour_of_day',
       'day_of_week', 'is_weekend', 'payment_method'],
      dtype='str')

# Connecting Jupyter to SSMS

In [57]:
# ============================================
# LOAD PANDAS DATAFRAME INTO SQL SERVER
# ============================================

!pip install -q pyodbc sqlalchemy pandas

import pandas as pd
from sqlalchemy import create_engine

# SQL Server details
server = r"DESKTOP-PT1D5QM\SQLEXPRESS"
database = "payment_failure_analysis"

# Create connection
connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

# Check that your DataFrame exists
print("Pandas dataset shape:", df.shape)

# Name of the SQL Server table
table_name = "payment_failure"

# Upload DataFrame to SQL Server
df.to_sql(
    table_name,
    con=engine,
    if_exists="replace",
    index=False
)

print(f"✅ Dataset successfully loaded into SQL Server!")
print(f"✅ Table created: {table_name}")
print(f"✅ Rows inserted: {len(df)}")

# Verify the table
check = pd.read_sql(
    f"SELECT TOP 5 * FROM [{table_name}]",
    engine
)

display(check)

Pandas dataset shape: (999, 18)
✅ Dataset successfully loaded into SQL Server!
✅ Table created: payment_failure
✅ Rows inserted: 999


,transaction_id,timestamp,transaction_type,merchant_category,amount_(INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,payment_method
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0,p2p
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0,p2m
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0,p2p
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1,p2p
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0,p2p
